In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("yolo11-no-c2psa.yaml"):
    print("FOUND:", p)

print("\n--- input top level ---")
for p in Path("/kaggle/input").iterdir():
    print("[DIR]", p.name)
    for f in list(p.rglob("*"))[:10]:
        print("   ", f.relative_to(p))

FOUND: /kaggle/input/datasets/roxyjiang12180318/ultralytics-cbam-project/ultralytics/cfg/models/11/yolo11-no-c2psa.yaml

--- input top level ---
[DIR] datasets
    roxyjiang12180318
    roxyjiang12180318/maizeleaf
    roxyjiang12180318/ultralytics-cbam-project
    roxyjiang12180318/maizeleaf/AAAA
    roxyjiang12180318/ultralytics-cbam-project/CONTRIBUTING.md
    roxyjiang12180318/ultralytics-cbam-project/README.zh-CN.md
    roxyjiang12180318/ultralytics-cbam-project/pyproject.toml
    roxyjiang12180318/ultralytics-cbam-project/README.md
    roxyjiang12180318/ultralytics-cbam-project/ultralytics
    roxyjiang12180318/ultralytics-cbam-project/mkdocs.yml


In [3]:
import sys, subprocess, shutil
from pathlib import Path

# 正确项目根目录
src = Path("/kaggle/input/datasets/roxyjiang12180318/ultralytics-cbam-project")
code_dir = Path("/kaggle/working/ultralytics-cbam-project")

# /kaggle/input 是只读的，先复制到可写目录
if not code_dir.exists():
    shutil.copytree(src, code_dir)

# 生成 Ultralytics AMP check 需要的占位图
import numpy as np, cv2
assets_dir = code_dir / "ultralytics" / "assets"
assets_dir.mkdir(parents=True, exist_ok=True)
cv2.imwrite(str(assets_dir / "bus.jpg"), np.ones((640, 640, 3), dtype=np.uint8) * 128)

# 用当前 notebook 的同一个 Python 安装
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", str(code_dir), "-q"],
    check=True,
)

# 手动加入 sys.path，避免 pip 刚装完但 editable 路径未立即生效
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

# 验证是否真的导入到我们的源码
import ultralytics
print("Python:", sys.executable)
print("Ultralytics version:", ultralytics.__version__)
print("Ultralytics path:", ultralytics.__file__)

from ultralytics import YOLO

yaml_path = code_dir / "ultralytics" / "cfg" / "models" / "11" / "yolo11-no-c2psa.yaml"
print("YAML exists:", yaml_path.exists())

model = YOLO(str(yaml_path))

results = model.train(
    data="/kaggle/input/datasets/roxyjiang12180318/maizeleaf/AAAA/data.yaml",
    epochs=200,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    patience=30,
    save=True,
    save_period=10,
    project="/kaggle/working/no-c2psa",
    name="exp",
    exist_ok=True,
    amp=True,
    cos_lr=True,
    warmup_epochs=3,
    pretrained=False,
)

print("Best model:", results.save_dir / "weights" / "best.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Python: /usr/bin/python3
Ultralytics version: 8.4.89
Ultralytics path: /kaggle/working/ultralytics-cbam-project/ultralytics/__init__.py
YAML exists: True
WARNING ⚠️ no model scale passed. Assuming scale='n'.
New https://pypi.org/project/ultralytics/8.4.115 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.89 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/

In [4]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display

exp_dir = Path("/kaggle/working/no-c2psa/exp")
zip_path = Path("/kaggle/working/no-c2psa_exp_results.zip")

print("结果目录存在:", exp_dir.exists())
print("包含文件:")
for f in sorted(exp_dir.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(exp_dir)}  ({f.stat().st_size / 1024:.1f} KB)")

# 删除旧的 zip，重新打包
if zip_path.exists():
    zip_path.unlink()
shutil.make_archive(str(zip_path.with_suffix("")), "zip", str(exp_dir))

print("\n打包完成:", zip_path)
print("大小:", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")

# 点击这个链接下载整个结果包
display(FileLink(str(zip_path)))

# 如果 zip 下载不顺利，单独下载 results.csv 备用
display(FileLink(str(exp_dir / "results.csv")))

结果目录存在: True
包含文件:
  BoxF1_curve.png  (160.0 KB)
  BoxPR_curve.png  (157.1 KB)
  BoxP_curve.png  (186.0 KB)
  BoxR_curve.png  (155.1 KB)
  args.yaml  (1.7 KB)
  confusion_matrix.png  (122.3 KB)
  confusion_matrix_normalized.png  (138.0 KB)
  labels.jpg  (181.4 KB)
  results.csv  (17.4 KB)
  results.png  (266.2 KB)
  train_batch0.jpg  (557.9 KB)
  train_batch1.jpg  (511.0 KB)
  train_batch2.jpg  (557.7 KB)
  val_batch0_labels.jpg  (528.8 KB)
  val_batch0_pred.jpg  (494.8 KB)
  val_batch1_labels.jpg  (570.5 KB)
  val_batch1_pred.jpg  (542.3 KB)
  val_batch2_labels.jpg  (530.2 KB)
  val_batch2_pred.jpg  (524.5 KB)
  weights/best.pt  (4846.5 KB)
  weights/epoch0.pt  (18769.7 KB)
  weights/epoch10.pt  (18770.7 KB)
  weights/epoch100.pt  (18781.9 KB)
  weights/epoch110.pt  (18783.2 KB)
  weights/epoch120.pt  (18784.4 KB)
  weights/epoch130.pt  (18785.7 KB)
  weights/epoch20.pt  (18771.9 KB)
  weights/epoch30.pt  (18773.2 KB)
  weights/epoch40.pt  (18774.4 KB)
  weights/epoch50.pt  (18775.7 K

/kaggle/working/no-c2psa_exp_results.zip

/kaggle/working/no-c2psa/exp/results.csv